# Road Accident Analysis — Data Cleaning
### Data Visualization Assignment

**Goal of this notebook:** load the raw RTA (Road Traffic Accident) dataset, understand what's in it, and clean it up so it's ready for visualization and analysis in the next step.

We'll go column by column and explain *why* each cleaning decision was made — not just what was done.

## 1. Load the dataset and take a first look

In [32]:
import pandas as pd
import numpy as np

df = pd.read_csv('RTA_Dataset.csv')

print("Shape (rows, columns):", df.shape)
df.head()

Shape (rows, columns): (12316, 32)


,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,Service_year_of_vehicle,...,Vehicle_movement,Casualty_class,Sex_of_casualty,Age_band_of_casualty,Casualty_severity,Work_of_casuality,Fitness_of_casuality,Pedestrian_movement,Cause_of_accident,Accident_severity
0,17:02:00,Monday,18-30,Male,Above high school,Employee,1-2yr,Automobile,Owner,Above 10yr,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Moving Backward,Slight Injury
1,17:02:00,Monday,31-50,Male,Junior high school,Employee,Above 10yr,Public (> 45 seats),Owner,5-10yrs,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury
2,17:02:00,Monday,18-30,Male,Junior high school,Employee,1-2yr,Lorry (41?100Q),Owner,NaN,...,Going straight,Driver or rider,Male,31-50,3,Driver,NaN,Not a Pedestrian,Changing lane to the left,Serious Injury
3,1:06:00,Sunday,18-30,Male,Junior high school,Employee,5-10yr,Public (> 45 seats),Governmental,NaN,...,Going straight,Pedestrian,Female,18-30,3,Driver,Normal,Not a Pedestrian,Changing lane to the right,Slight Injury
4,1:06:00,Sunday,18-30,Male,Junior high school,Employee,2-5yr,NaN,Owner,5-10yrs,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury


In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12316 entries, 0 to 12315
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Time                         12316 non-null  str  
 1   Day_of_week                  12316 non-null  str  
 2   Age_band_of_driver           12316 non-null  str  
 3   Sex_of_driver                12316 non-null  str  
 4   Educational_level            11575 non-null  str  
 5   Vehicle_driver_relation      11737 non-null  str  
 6   Driving_experience           11487 non-null  str  
 7   Type_of_vehicle              11366 non-null  str  
 8   Owner_of_vehicle             11834 non-null  str  
 9   Service_year_of_vehicle      8388 non-null   str  
 10  Defect_of_vehicle            7889 non-null   str  
 11  Area_accident_occured        12077 non-null  str  
 12  Lanes_or_Medians             11931 non-null  str  
 13  Road_allignment              12174 non-null  str  
 14  T

**Observation:** we have 12,316 rows and 32 columns. Most columns are categorical text (like `Weather_conditions`, `Cause_of_accident`), and a few are numeric (`Number_of_vehicles_involved`, `Number_of_casualties`). `Time` is currently just text, not a real time value yet.

## 2. Check for missing values — the naive way

In [21]:
df.replace('na', np.nan, inplace=True)

missing_count = df.isnull().sum()
print(missing_count[missing_count > 0])

Educational_level           741
Vehicle_driver_relation     579
Driving_experience          829
Type_of_vehicle             950
Owner_of_vehicle            482
Service_year_of_vehicle    3928
Defect_of_vehicle          4427
Area_accident_occured       239
Lanes_or_Medians            385
Road_allignment             142
Types_of_Junction           887
Road_surface_type           172
Type_of_collision           155
Vehicle_movement            308
Casualty_class             4443
Sex_of_casualty            4443
Age_band_of_casualty       4443
Casualty_severity          4443
Work_of_casuality          3198
Fitness_of_casuality       2635
dtype: int64


**Observation:** at first glance, only 15 columns have missing values, and columns like `Casualty_class`, `Sex_of_casualty`, `Age_band_of_casualty`, and `Casualty_severity` look perfectly complete (0% missing).

**But this is misleading.** Let's check what values those "complete" columns actually contain.

In [22]:
df['Casualty_class'].unique()

<StringArray>
[nan, 'Driver or rider', 'Pedestrian', 'Passenger']
Length: 4, dtype: str

**Important finding:** the value `'na'` appears as literal *text*, not as a real missing value (`NaN`). Pandas' `isna()` only detects true blanks, so it completely missed this. Let's check how common this is.

In [23]:
for col in ['Casualty_class', 'Sex_of_casualty', 'Age_band_of_casualty', 'Casualty_severity']:
    na_count = (df[col] == 'na').sum()
    print(f"{col}: {na_count} rows ({na_count/len(df)*100:.1f}%) are the text 'na'")

Casualty_class: 0 rows (0.0%) are the text 'na'
Sex_of_casualty: 0 rows (0.0%) are the text 'na'
Age_band_of_casualty: 0 rows (0.0%) are the text 'na'
Casualty_severity: 0 rows (0.0%) are the text 'na'


**Insight:** these 4 columns are actually **~36% missing**, not 0%. This is exactly why you always inspect a dataset's actual values, not just rely on automatic missing-value counts — a dataset can hide missingness behind placeholder text.

We'll fix this in the next step by converting `'na'` strings into real `NaN` values everywhere in the dataset.

## 3. Fix the hidden 'na' strings

In [24]:
# Replace the literal text 'na' with a proper missing value (NaN) across the whole dataframe
df.replace('na', np.nan, inplace=True)

# Recheck missing percentages now that 'na' strings are real NaNs
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct[missing_pct > 0]

Sex_of_casualty            36.1
Age_band_of_casualty       36.1
Casualty_severity          36.1
Casualty_class             36.1
Defect_of_vehicle          35.9
Service_year_of_vehicle    31.9
Work_of_casuality          26.0
Fitness_of_casuality       21.4
Type_of_vehicle             7.7
Types_of_Junction           7.2
Driving_experience          6.7
Educational_level           6.0
Vehicle_driver_relation     4.7
Owner_of_vehicle            3.9
Lanes_or_Medians            3.1
Vehicle_movement            2.5
Area_accident_occured       1.9
Road_surface_type           1.4
Type_of_collision           1.3
Road_allignment             1.2
dtype: float64

**Observation:** now the true picture is visible. We have three tiers of missingness:
- **Heavy (20–36%):** `Defect_of_vehicle`, `Service_year_of_vehicle`, `Work_of_casuality`, `Fitness_of_casuality`, and the 4 casualty columns we just uncovered
- **Moderate (2–8%):** `Type_of_vehicle`, `Types_of_Junction`, `Driving_experience`, etc.
- **Light (<2%):** `Road_surface_type`, `Type_of_collision`, `Road_allignment`

Each tier needs a different treatment — that's what we do next.

## 4. Handle heavy-missing columns

For `Defect_of_vehicle`, `Service_year_of_vehicle`, `Work_of_casuality`, and `Fitness_of_casuality`: these are **vehicle/casualty maintenance details that aren't central to our main question** (what causes accidents and how severe they are). Since over a fifth of the data is missing for each, filling them in would mostly be guessing. We drop these columns entirely rather than drop rows — dropping rows would cost us usable data in every *other* column too.

In [25]:
cols_to_drop = ['Defect_of_vehicle', 'Service_year_of_vehicle',
                'Work_of_casuality', 'Fitness_of_casuality']

df.drop(columns=cols_to_drop, inplace=True)
print("Remaining columns:", df.shape[1])

Remaining columns: 28


## 5. Handle moderate + the newly-found heavy-missing categorical columns

For the rest of the missing categorical columns — including the 4 casualty columns we uncovered in Step 2 — we **fill missing values with `'Unknown'`** instead of dropping rows.

Why fill instead of drop? Because dropping rows for the casualty columns alone would remove about 36% of our entire dataset — far too costly. Treating "missing" as its own category ("Unknown") keeps all the rows and is honest about the fact that this information simply wasn't recorded, rather than assuming it was one specific value.

In [26]:
fill_unknown_cols = [
    'Type_of_vehicle', 'Types_of_Junction', 'Driving_experience', 'Educational_level',
    'Vehicle_driver_relation', 'Owner_of_vehicle', 'Lanes_or_Medians', 'Vehicle_movement',
    'Area_accident_occured',
    'Casualty_class', 'Sex_of_casualty', 'Age_band_of_casualty', 'Casualty_severity'
]

for col in fill_unknown_cols:
    df[col] = df[col].fillna('Unknown')

print("Missing values left in these columns:", df[fill_unknown_cols].isna().sum().sum())

Missing values left in these columns: 0


## 6. Handle the lightly-missing columns

For `Road_surface_type`, `Type_of_collision`, and `Road_allignment`, less than 2% of rows are missing. At this small a scale, it's simpler and safe to just **drop those specific rows** — we won't lose meaningful information, and it avoids adding more "Unknown" categories where we don't need them.

In [27]:
before = len(df)

df.dropna(subset=['Road_surface_type', 'Type_of_collision', 'Road_allignment'], inplace=True)

print(f"Rows before: {before}, rows after: {len(df)}  (dropped {before - len(df)} rows, "
      f"{(before-len(df))/before*100:.1f}%)")

Rows before: 12316, rows after: 11853  (dropped 463 rows, 3.8%)


## 7. Fix the `Time` column

`Time` is currently stored as plain text (e.g. `17:02:00`). Since "what time of day do accidents happen" is one of our analytical questions, we convert it to a real datetime type and pull out the **hour** as its own numeric column — that's the piece we'll actually plot later.

In [28]:
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S')
df['Hour'] = df['Time'].dt.hour

df[['Time', 'Hour']].head()

,Time,Hour
0,1900-01-01 17:02:00,17
1,1900-01-01 17:02:00,17
3,1900-01-01 01:06:00,1
4,1900-01-01 01:06:00,1
7,1900-01-01 17:20:00,17


## 8. Check for duplicate records

In [29]:
dup_count = df.duplicated().sum()
print(f"Duplicate rows found: {dup_count}")

Duplicate rows found: 0


**Observation:** no duplicate rows — nothing to remove here.

## 9. Final check

In [30]:
print("Final shape:", df.shape)
print(f"Rows kept: {len(df)} out of 12,316 original ({len(df)/12316*100:.1f}%)")
print("Total missing values remaining:", df.isna().sum().sum())

df.head()

Final shape: (11853, 29)
Rows kept: 11853 out of 12,316 original (96.2%)
Total missing values remaining: 0


,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,Area_accident_occured,...,Number_of_casualties,Vehicle_movement,Casualty_class,Sex_of_casualty,Age_band_of_casualty,Casualty_severity,Pedestrian_movement,Cause_of_accident,Accident_severity,Hour
0,1900-01-01 17:02:00,Monday,18-30,Male,Above high school,Employee,1-2yr,Automobile,Owner,Residential areas,...,2,Going straight,Unknown,Unknown,Unknown,Unknown,Not a Pedestrian,Moving Backward,Slight Injury,17
1,1900-01-01 17:02:00,Monday,31-50,Male,Junior high school,Employee,Above 10yr,Public (> 45 seats),Owner,Office areas,...,2,Going straight,Unknown,Unknown,Unknown,Unknown,Not a Pedestrian,Overtaking,Slight Injury,17
3,1900-01-01 01:06:00,Sunday,18-30,Male,Junior high school,Employee,5-10yr,Public (> 45 seats),Governmental,Office areas,...,2,Going straight,Pedestrian,Female,18-30,3,Not a Pedestrian,Changing lane to the right,Slight Injury,1
4,1900-01-01 01:06:00,Sunday,18-30,Male,Junior high school,Employee,2-5yr,Unknown,Owner,Industrial areas,...,2,Going straight,Unknown,Unknown,Unknown,Unknown,Not a Pedestrian,Overtaking,Slight Injury,1
7,1900-01-01 17:20:00,Friday,18-30,Male,Junior high school,Employee,2-5yr,Automobile,Governmental,Residential areas,...,1,U-Turn,Unknown,Unknown,Unknown,Unknown,Not a Pedestrian,No priority to vehicle,Slight Injury,17


**Summary of what we did and why:**

| Step | Action | Reason |
|---|---|---|
| Hidden `'na'` strings | Converted to real `NaN` | Prevented us from under-counting missing data (uncovered 36% missingness hiding in 4 "complete-looking" columns) |
| 4 heavy-missing columns | Dropped columns | Not central to the accident-cause/severity question; too incomplete to fill reliably |
| 13 moderate/heavy categorical columns | Filled with `'Unknown'` | Preserves all rows; honest about missing info rather than guessing a value |
| 3 lightly-missing columns | Dropped the few affected rows | Minimal data loss (<2%), simplest fix |
| `Time` | Converted to datetime, extracted `Hour` | Needed for time-of-day analysis question |
| Duplicates | Checked, none found | N/A |

We kept 96%+ of the original rows while cleaning up every column we plan to analyze. The cleaned dataframe `df` is now ready for the next notebook: formulating analytical questions and building visualizations.

In [31]:
df.to_csv('RTA_Dataset_cleaned.csv', index=False)
print("Saved cleaned dataset:", df.shape)

Saved cleaned dataset: (11853, 29)
